# Project · SpecSense AI — explainable product recommendations

Free-text shopping need → hard specs + soft preferences → explainable recommendation. First the **original** ReAct pipeline (answers from model memory), then the **catalog-grounded** version (extract → match with pandas → explain) with gap detection.

> 🔐 **Keys** — nothing in this notebook contains a key. In Colab open the **🔑 Secrets** panel (left sidebar), click **+ Add new secret**,
> type the *name* exactly as below, paste the *value*, and switch on **Notebook access**. If you skip this, you'll be asked once (hidden input).

| Secret name | What it is | Get it at |
|---|---|---|
| `GEMINI_API_KEY` | Google Gemini API key | https://aistudio.google.com/apikey |

**Never** paste a key into a code cell, commit it to GitHub, or share a notebook that shows it in an output. If one leaks, delete it in the provider's console and create a new one.

In [ ]:
!pip install -q google-genai pandas pydantic python-dotenv

In [ ]:
import os, getpass

def get_secret(name, *, required=True, hint=""):
    """1) environment  2) Colab Secrets (key icon in the left sidebar)  3) hidden prompt.
    The value is kept in memory / os.environ only - it is never printed or saved in the notebook."""
    val = os.getenv(name)
    if not val:
        try:
            from google.colab import userdata          # works only inside Colab
            val = userdata.get(name)
        except Exception:
            val = None
    if not val and required:
        val = getpass.getpass(f"Paste {name} {hint}(input is hidden): ").strip()
    if val:
        os.environ[name] = val
    return val

## 1 · Create the project files in this Colab session
Each cell writes one file (`%%writefile`). `catalog_matcher.py` and the sample catalog are the grounded add-on.

In [ ]:
%cd /content
!mkdir -p specsense_ai
%cd specsense_ai

In [ ]:
import os
for d in ['data', 'llm', 'llm/prompts', 'pipeline', 'pipeline/spec_sensor']:
    os.makedirs(d, exist_ok=True)

In [ ]:
%%writefile utils.py
import logging
import re
import json
from pathlib import Path

#Configures logging to show timestamp, severity, and message.
def get_logger(name: str) -> logging.Logger:
    """Get a simple logger for the given name."""
    logging.basicConfig(
        level=logging.INFO,
        format='[%(asctime)s] %(levelname)s: %(message)s'
    )
    return logging.getLogger(name)

def extract_json_from_text(text: str) -> dict:
    """Extract JSON from text string. Returns empty dict if not found."""
    try:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if match:
            return json.loads(match.group())
        return {}
    except json.JSONDecodeError:
        return {}

def load_file(file_path: str) -> str:
    """Load and return file contents as string."""
    return Path(file_path).read_text().strip()

In [ ]:
%%writefile config.py
import os
from pathlib import Path
from dotenv import load_dotenv

# Load environment variables
PROJECT_ROOT = Path(__file__).parent
load_dotenv(PROJECT_ROOT / ".env")

# API Configuration
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")   # from .env - never hard-code

# LLM Settings
DEFAULT_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.6-flash")   # e.g. gemini-2.5-flash or gemini-3.5-flash-lite
MAX_RETRIES = 3
RETRY_DELAY = 2

# Processing Settings
DEFAULT_BATCH_SIZE = 10
STREAMLIT_BATCH_SIZE = 5

In [ ]:
%%writefile llm/__init__.py

In [ ]:
%%writefile llm/client.py
import time
from google import genai
from config import GEMINI_API_KEY, DEFAULT_MODEL, MAX_RETRIES, RETRY_DELAY
from utils import get_logger

logger = get_logger(__name__)

class LLMClient:
   """Gemini API client"""  
   def __init__(self, model_name=DEFAULT_MODEL, max_retries=MAX_RETRIES, retry_delay=RETRY_DELAY):
       self.model_name = model_name
       self.max_retries = max_retries
       self.retry_delay = retry_delay
       self.client = genai.Client(api_key=GEMINI_API_KEY)

   def call(self, prompt: str, **kwargs) -> str:
       """Send prompt to Gemini API"""
       for attempt in range(self.max_retries + 1):
           try:
               response = self.client.models.generate_content(
                   contents=prompt,
                   model=self.model_name,
                   #vertex="text",
                   **kwargs                  
               )
               if response and response.text:
                   return response.text.strip()
               else:
                   raise Exception("Empty response received")

           except Exception as e:
               if attempt == self.max_retries:
                   raise Exception(f"API call failed after {self.max_retries + 1} attempts: {e}")

               time.sleep(self.retry_delay * (2 ** attempt))

In [ ]:
%%writefile llm/prompts.py
from pathlib import Path
from utils import load_file

# Directory paths
PROMPTS_DIR = Path(__file__).parent / "prompts"

# Load prompt templates from external files
def load_prompt(filename: str) -> str:
    """Load a prompt template from file."""
    return load_file(PROMPTS_DIR / filename)

# Available prompt templates
PROMPT = load_prompt("react.md")  # <-- Edit this to use different prompts

def generate_prompt(user_input: str) -> str:
    """Generate a full prompt by combining template and user input."""
    template = PROMPT
    return f"{template}\n\nUser Message:\n{user_input.strip()}"

In [ ]:
%%writefile llm/prompts/react.md
You are a highly reliable and transparency-focused AI system trained to interpret natural language product queries and recommend items that best match both hard specifications and soft preferences.

Follow this exact structured reasoning format for each query:

1. **Thought**: Analyze the user’s query for budget, category, and preferences.
2. **Action**: Extract hard constraints (strict specs like price, category, RAM, weight) and soft preferences (subjective needs like portability, camera quality, eco-friendliness).
3. **Observation**: Match products against these requirements, noting where they fit or fall short.
4. **Final Answer**: Output a structured JSON with the following fields:

```json
{
  "recommended_product": "<main product name>",
  "meets": ["<criteria fully met>"],
  "partially_meets": ["<criteria partially met>"],
  "trade_offs": ["<compromises or limitations>"],
  "review_snippets": ["<relevant customer review quotes>"],
  "alternatives": ["<other product options if any>"]
}
```

Rules:
- Do not include Thought, Action, or Observation text.
- Always explain what is met, partially met, and trade‑offs.
- Include at least one review snippet if available.
- If no perfect match, suggest nearest alternatives.

In [ ]:
%%writefile pipeline/__init__.py

In [ ]:
%%writefile pipeline/spec_sensor/__init__.py

In [ ]:
%%writefile pipeline/spec_sensor/builder.py
from llm.prompts import generate_prompt

def build_prompt(message: str, strategy: str = "react") -> str:
    """Build a prompt using the chosen strategy."""
    if strategy == "react":
        return generate_prompt(message)
    else:
        raise NotImplementedError(f"Strategy '{strategy}' is not supported yet.")

In [ ]:
%%writefile pipeline/spec_sensor/executor.py
from typing import Optional
from llm.client import LLMClient
from utils import get_logger

logger = get_logger(__name__)

class LLMExecutor:
    """Executes prompts using the LLM client."""
    def __init__(self, model: Optional[str] = None) -> None:
        self.llm: LLMClient = LLMClient(model) if model else LLMClient()
        logger.info("Initialized LLMExecutor")

    def execute(self, prompt: str) -> str:
        logger.info(f"Executing LLM with prompt length: {len(prompt)}")
        try:
            response = self.llm.call(prompt)
            logger.info(f"LLM execution successful, response length: {len(response)}")
            return response
        except Exception as e:
            logger.error(f"LLM execution failed: {str(e)}")
            raise

In [ ]:
%%writefile pipeline/spec_sensor/parser.py
from typing import Dict, Any
from utils import get_logger, extract_json_from_text

logger = get_logger(__name__)

class OutputParser:
    """Parses LLM output into structured format for SpecSense AI."""

    def parse_llm_output(self, llm_output: Any) -> Dict[str, Any]:
        logger.info(f"Parsing LLM output of type {type(llm_output)}")

        # Case 1: None
        if llm_output is None:
            logger.error("LLM output is None.")
            return {
                "recommended_product": "Unknown",
                "meets": [],
                "partially_meets": [],
                "trade_offs": [],
                "review_snippets": [],
                "alternatives": []
            }

        # Case 2: Gemini structured response
        if hasattr(llm_output, "candidates"):
            try:
                parts = llm_output.candidates[0].content.parts
                text_parts = [p.text for p in parts if hasattr(p, "text") and p.text]
                llm_text = "\n".join(text_parts) if text_parts else ""
                logger.info(f"Extracted structured LLM output, text length: {len(llm_text)}")
            except Exception as e:
                logger.error(f"Failed to extract text from Gemini response: {e}")
                llm_text = ""
        else:
            # Case 3: Plain string
            llm_text = str(llm_output)
            logger.info(f"Parsing plain LLM output of length: {len(llm_text)}")

        # Try to extract JSON
        parsed_json = extract_json_from_text(llm_text)

        if parsed_json:
            logger.info("Successfully parsed LLM output to JSON.")
            return {
                "recommended_product": parsed_json.get("recommended_product", "Unknown"),
                "meets": parsed_json.get("meets", []),
                "partially_meets": parsed_json.get("partially_meets", []),
                "trade_offs": parsed_json.get("trade_offs", []),
                "review_snippets": parsed_json.get("review_snippets", []),
                "alternatives": parsed_json.get("alternatives", [])
            }
        else:
            logger.warning("No JSON found in LLM output.")
            return {
                "recommended_product": "Unknown",
                "meets": [],
                "partially_meets": [],
                "trade_offs": [],
                "review_snippets": [],
                "alternatives": []
            }

In [ ]:
%%writefile pipeline/spec_sensor/sensor.py
from typing import List, Dict, Any
from .builder import build_prompt
from .executor import LLMExecutor
from .parser import OutputParser
from utils import get_logger

logger = get_logger(__name__)

class SpecSensor:
    """Orchestrates the spec sensor pipeline."""

    def __init__(self, strategy: str = "react") -> None:
        """Initializes the pipeline components."""
        self.executor = LLMExecutor()
        self.parser = OutputParser()
        self.strategy = strategy
        logger.info(f"Initialized spec sensor with strategy: {self.strategy}")

    def detect(self, message: str) -> Dict[str, Any]:
        """Runs spec sensor on a single query: build prompt -> execute -> parse."""
        logger.info(f"Starting spec sensor for message length: {len(message)}")
        try:
            # The 3-step pipeline
            prompt = build_prompt(message, self.strategy)
            raw_response = self.executor.execute(prompt)
            parsed_result = self.parser.parse_llm_output(raw_response)

            logger.info(f"Spec sensor run is successful. Recommended: {parsed_result.get('recommended_product', 'Unknown')}")
            return parsed_result

        except Exception as e:
            logger.error(f"spec sensor pipeline failed: {e}")
            # Re-raise the exception to be handled by the caller (UI layer)
            raise

    def detect_batch(self, messages: List[str]) -> List[Dict[str, Any]]:
        """Runs spec sensor on a list of messages."""
        total_messages = len(messages)
        logger.info(f"Starting batch spec sensor for {total_messages} messages.")

        results: List[Dict[str, Any]] = []

        for i, message in enumerate(messages):
            try:
                result = self.detect(message)
                results.append(result)
            except Exception as e:
                logger.warning(f"Failed to process message {i+1}/{total_messages}: {e}")
                # Append a fallback error result for the failed message
                error_result = {
                    "recommended_product": "Unknown",
                    "meets": [],
                    "partially_meets": [],
                    "trade_offs": [f"processing_error: {e}"],
                    "review_snippets": [],
                    "alternatives": []
                }
                results.append(error_result)

        successful = sum(1 for r in results if r.get("recommended_product") != "Unknown")
        logger.info(f"Batch processing complete. {successful}/{total_messages} succeeded.")
        return results

In [ ]:
%%writefile catalog_matcher.py
"""Catalog-grounded SpecSense (the "intelligence layer" from the problem statement).

The original pipeline (main.py / streamlit/app.py) asks the model to recommend a product from MEMORY - so the product,
its specs and the "customer reviews" can be invented. This module fixes that in three steps:

    1. EXTRACT   LLM turns the free-text query into hard constraints + soft preferences (JSON)
    2. MATCH     plain pandas filters the company catalog (exact, testable, no hallucination);
                 if nothing fits, it returns the products that break the fewest constraints (gap detection)
    3. EXPLAIN   LLM writes the recommendation using ONLY the retrieved rows (meets / partially / trade-offs / review / alternatives)

Run:  python catalog_matcher.py "Lightweight laptop under 30000 for coding, good battery"
"""
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd

CATALOG_PATH = Path(__file__).parent / "data" / "products_sample.csv"

EXTRACT_PROMPT = """Extract the shopper's needs from the query. Return ONLY JSON with these keys (use null when not mentioned):
{"max_price_inr": number|null, "max_weight_kg": number|null, "min_ram_gb": number|null, "min_storage_gb": number|null,
 "min_battery_hours": number|null, "soft_preferences": ["short phrases such as 'good for coding', 'travel friendly'"]}
Translate words into numbers when obvious (e.g. 'under 30k' -> 30000, 'lightweight' -> max_weight_kg 1.5).

Query:
"""

EXPLAIN_PROMPT = """You are a transparent product advisor. Recommend using ONLY the catalog rows provided - never invent a product,
a spec or a review. Use the row's `review` text as the review snippet.
Return ONLY JSON:
{"recommended_product": "<name>", "meets": ["hard criteria fully met"], "partially_meets": ["soft preferences only partly met"],
 "trade_offs": ["compromises"], "review_snippets": ["quote from the row's review"], "alternatives": ["other catalog names + one-line reason"],
 "gap_note": "<empty string if everything is met, else what could not be met>"}
"""

HARD = {  # constraint -> (column, comparison)
    "max_price_inr": ("price_inr", "<="), "max_weight_kg": ("weight_kg", "<="), "min_ram_gb": ("ram_gb", ">="),
    "min_storage_gb": ("storage_gb", ">="), "min_battery_hours": ("battery_hours", ">="),
}


def load_catalog(path: Path = CATALOG_PATH) -> pd.DataFrame:
    return pd.read_csv(path)


def violations(row: pd.Series, constraints: dict) -> list[str]:
    """Which hard constraints does this product break? (empty list = fully matches)"""
    out = []
    for key, (col, op) in HARD.items():
        want = constraints.get(key)
        if want is None:
            continue
        ok = row[col] <= want if op == "<=" else row[col] >= want
        if not ok:
            out.append(f"{col} {row[col]} (wanted {op} {want})")
    return out


def shortfall(row: pd.Series, constraints: dict) -> float:
    """How far (relative) the product misses its violated constraints - used to rank near-misses."""
    total = 0.0
    for key, (col, op) in HARD.items():
        want = constraints.get(key)
        if want and ((row[col] > want) if op == "<=" else (row[col] < want)):
            total += abs(row[col] - want) / want
    return round(total, 3)


def match(df: pd.DataFrame, constraints: dict, top_k: int = 3) -> tuple[pd.DataFrame, bool]:
    """Return (rows, exact). exact=False means no product met every hard constraint, so these are the nearest."""
    df = df[df["in_stock"] == "yes"].copy()
    df["violations"] = df.apply(lambda r: violations(r, constraints), axis=1)
    df["n_viol"] = df["violations"].map(len)
    df["shortfall"] = df.apply(lambda r: shortfall(r, constraints), axis=1)
    best = df["n_viol"].min()
    exact = best == 0
    if exact:
        rows = df[df["n_viol"] == 0].sort_values("price_inr").head(top_k)
    else:  # no perfect match: show the nearest misses (fewest broken constraints, then smallest shortfall)
        rows = df.sort_values(["n_viol", "shortfall", "price_inr"]).head(top_k)
    return rows, exact


def recommend(query: str, llm=None) -> dict:
    from llm.client import LLMClient          # imported here so the pure-pandas helpers above work without an API key
    from utils import extract_json_from_text
    llm = llm or LLMClient()
    constraints = extract_json_from_text(llm.call(EXTRACT_PROMPT + query)) or {}
    rows, exact = match(load_catalog(), constraints)
    payload = rows.drop(columns=["n_viol", "shortfall"]).to_dict(orient="records")
    prompt = (f"{EXPLAIN_PROMPT}\nUser query: {query}\nExtracted constraints: {json.dumps(constraints)}\n"
              f"Exact match found: {exact}\nCatalog rows (the ONLY products you may mention):\n{json.dumps(payload, ensure_ascii=False)}")
    result = extract_json_from_text(llm.call(prompt))
    result["_constraints"], result["_exact"], result["_candidates"] = constraints, exact, rows["name"].tolist()
    return result


if __name__ == "__main__":
    q = " ".join(sys.argv[1:]) or "Lightweight laptop under 30000 for coding, good battery"
    print(json.dumps(recommend(q), indent=2, ensure_ascii=False))

In [ ]:
%%writefile data/products_sample.csv
id,name,category,price_inr,weight_kg,ram_gb,storage_gb,cpu,battery_hours,in_stock,offer,description,review
L01,Nimbus Air 14 Lite,laptop,27990,1.30,8,256,Ryzen 3 class,9,yes,₹1000 off with bank card,Thin 14-inch student laptop with a bright IPS screen,"Light enough to carry all day and VS Code runs fine, but the screen could be sharper."
L02,Nimbus Air 14 Plus,laptop,32990,1.30,16,512,Ryzen 5 class,10,yes,none,Same body as the Lite with double the memory and storage,"16 GB RAM makes multitasking smooth; battery lasts a full college day."
L03,Vertex Student 15,laptop,24990,1.75,8,512,Core i3 class,6,yes,Free sleeve,Budget 15-inch laptop with a roomy keyboard,"Great value and a comfortable keyboard, but it is bulky and the battery is average."
L04,Vertex Pro 14,laptop,58990,1.45,16,512,Core i5 class,12,yes,none,Business ultraportable with a metal body,"Fast, quiet, and the battery easily lasts a work day."
L05,Orbit Creator 16,laptop,89990,2.10,32,1024,Core i7 class + RTX GPU,6,no,none,Creator laptop with a colour-accurate OLED screen,"Superb screen and speed for video editing; heavy and the fans get loud."
L06,Orbit Go 13,laptop,29490,1.10,8,256,Core i3 class,11,yes,Extra 6 months warranty,Ultra-light 13-inch laptop built for travel,"Barely notice it in my bag and the battery goes on forever; only 256 GB storage."
L07,Lumo Code 14,laptop,41990,1.35,16,512,Ryzen 5 class,13,yes,none,Developer-friendly 14-inch with upgradable RAM and a great keyboard,"Compiles quickly, keyboard is excellent and the battery lasts two days of classes."
L08,Lumo Basic 14,laptop,21990,1.60,4,128,Pentium class,5,yes,none,Entry-level laptop for browsing and documents,"Fine for web and documents; slow if you open many tabs."
L09,Nimbus Gamer 15,laptop,74990,2.30,16,512,Ryzen 7 class + RTX GPU,5,yes,Free headset,Gaming laptop with a 144 Hz display,"Smooth games, but it is heavy and the battery is short."
L10,Vertex Slate 12,laptop,35990,0.95,8,256,Core m class,14,yes,none,Ultra-compact 12-inch with all-day battery,"Incredibly light and long battery; underpowered for heavy coding."
L11,Orbit Edu 14,laptop,26990,1.40,8,256,Core i3 class,8,yes,Free mouse,Rugged laptop designed for schools,"Survived being dropped; battery is decent and it is easy to repair."
L12,Lumo Studio 15,laptop,64990,1.80,16,1024,Ryzen 7 class,9,no,none,Large-screen laptop for design and analytics,"Gorgeous large display; sold out most weeks."

## 2 · Key

In [ ]:
import os, getpass

def get_secret(name, *, required=True, hint=""):
    """1) environment  2) Colab Secrets (key icon in the left sidebar)  3) hidden prompt.
    The value is kept in memory / os.environ only - it is never printed or saved in the notebook."""
    val = os.getenv(name)
    if not val:
        try:
            from google.colab import userdata          # works only inside Colab
            val = userdata.get(name)
        except Exception:
            val = None
    if not val and required:
        val = getpass.getpass(f"Paste {name} {hint}(input is hidden): ").strip()
    if val:
        os.environ[name] = val
    return val

get_secret("GEMINI_API_KEY", hint="(from aistudio.google.com/apikey) ")
os.environ["GEMINI_MODEL"] = "gemini-2.5-flash"   # or "gemini-3.5-flash-lite" if your key supports it

## 3 · Original pipeline (recommends from the model's memory)
Look at `review_snippets` and the product name — then compare with the catalog in section 5.

In [ ]:
import sys, json; sys.path.insert(0, "/content/specsense_ai")
from pipeline.spec_sensor.sensor import SpecSensor
sensor = SpecSensor()
q = "Find an affordable, lightweight laptop under rupees 30k suitable for coding and programming students."
orig = sensor.detect(q)
print(json.dumps(orig, indent=2, ensure_ascii=False))

## 4 · The catalog (12 fictional laptops)

In [ ]:
import pandas as pd
cat = pd.read_csv('data/products_sample.csv')
cat.drop(columns=['description'])

In [ ]:
name = orig.get("recommended_product", "")
print("Recommended by the original pipeline:", name)
print("Exists in our catalog?", any(name.lower() == n.lower() for n in cat["name"]))

## 5 · Grounded version: extract → match → explain
The matching step is plain pandas, so it is exact and testable; the LLM only extracts needs and explains the retrieved rows.

In [ ]:
from catalog_matcher import recommend
res = recommend("Lightweight laptop under 30000 for coding students, good battery backup")
print("Constraints :", res.pop("_constraints")); print("Exact match :", res.pop("_exact")); print("Candidates  :", res.pop("_candidates"), "\n")
print(json.dumps(res, indent=2, ensure_ascii=False))

## 6 · Gap detection
Ask for something the catalog cannot satisfy: under ₹30,000 **and** 16 GB RAM.

In [ ]:
import catalog_matcher as cm
rows, exact = cm.match(cm.load_catalog(), {"max_price_inr": 30000, "min_ram_gb": 16})
print("Exact match?", exact)
rows[["name", "price_inr", "ram_gb", "violations"]]

## 7 · Automatic checks (no LLM)
Is the product in the catalog? Are the hard constraints met? Is the review quote verbatim?

In [ ]:
def check(rec, limits):
    row = cat[cat.name == rec.get("recommended_product")]
    exists = len(row) == 1
    ok = exists and row.price_inr.iloc[0] <= limits["price_inr"] and row.weight_kg.iloc[0] <= limits["weight_kg"]
    quote = exists and all(sn.lower() in row.review.iloc[0].lower() for sn in rec.get("review_snippets", []))
    return {"in catalog": exists, "hard constraints met": ok, "review verbatim": quote}
lim = {"price_inr": 30000, "weight_kg": 1.5}
print("original :", check(orig, lim)); print("grounded :", check(res, lim))

### Try next
* Replace `data/products_sample.csv` with your own catalog (same columns).
* Add a `min_screen_inches` constraint in `HARD` and the extract prompt.
* Ask the LLM judge (see the app's SpecSense page) to score explainability.